In [1]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    print(dirname)
    for f in filenames:
        print("   ", f)

/kaggle/input
/kaggle/input/competitions
/kaggle/input/competitions/aisehack-2-0
    sample_submission.csv
    base_line_model.ipynb
    train.csv
    test.csv


In [2]:
import pandas as pd

path = "/kaggle/input/competitions/aisehack-2-0"

train = pd.read_csv(path + "/train.csv")
test = pd.read_csv(path + "/test.csv")
sample = pd.read_csv(path + "/sample_submission.csv")

print(train.shape)
print(test.shape)
print(sample.shape)

train.head()

(6171, 3)
(4115, 3)
(10, 2)


,smiles,target,target_type
0,*Oc1ccc(cc1)C1(c2cc(ccc2c2ccc(cc12)[N+](=O)[O-...,294.0000,tg
1,*CCCCOC(=O)NC1CCC(CC2CCC(NC(=O)O*)CC2)CC1,6.1232,egc
2,*N1C(=O)c2c(C1=O)cc(cc2)Oc1cc2c(cc1Oc1cc3c(C(=...,222.5000,tg
3,*N1C(=O)c2c(C1=O)cc(cc2)S(=O)(=O)c1cc2c(C(=O)N...,72.0000,tg
4,*N1C(=O)c2c(C1=O)cc(cc2)C(c1cc2c(C(=O)N(C2=O)C...,224.0000,tg


In [3]:
train.info()

print("\nUnique target types:")
print(train["target_type"].value_counts())

print("\nMissing values:")
print(train.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6171 entries, 0 to 6170
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   smiles       6171 non-null   object 
 1   target       6171 non-null   float64
 2   target_type  6171 non-null   object 
dtypes: float64(1), object(2)
memory usage: 144.8+ KB

Unique target types:
target_type
tg     4143
egc    2028
Name: count, dtype: int64

Missing values:
smiles         0
target         0
target_type    0
dtype: int64


In [4]:
test.head()

,id,smiles,target_type
0,1,*N1C(=O)c2c(C1=O)cc(cc2)NC(=O)Nc1c2c(ccc1)c(cc...,tg
1,2,*Oc1ccc(C(C)(C)c2cc(Cl)c(OC(*)=O)c(Cl)c2)cc1,egc
2,3,*C1C(CC(C1)C=C*)(C[Si](O[Si](C)(C)C)(C)C)C#N,tg
3,4,*NC(=O)CCCCCCCCCCCCCCCCCCC(=O)NCCCCCCCCCCC*,tg
4,5,*C(C*)(C(=O)OCCCCCCN1CCN(CC1)c1ccc(cc1)N=Nc1cc...,tg


In [5]:
sample.head()

,id,target
0,1,273.5
1,2,195.0
2,3,44.0
3,4,45.0
4,5,67.0


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 6),
    max_features=15000,
    lowercase=False,
    sublinear_tf=True
)
X_train = vectorizer.fit_transform(train["smiles"])
X_test = vectorizer.transform(test["smiles"])

print("Train Features:", X_train.shape)
print("Test Features:", X_test.shape)

Train Features: (6171, 15000)
Test Features: (4115, 15000)


In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Split the training data
tg_train = train[train["target_type"] == "tg"]
egc_train = train[train["target_type"] == "egc"]

# Create TF-IDF features separately
X_tg = vectorizer.transform(tg_train["smiles"])
X_egc = vectorizer.transform(egc_train["smiles"])

y_tg = tg_train["target"]
y_egc = egc_train["target"]

# Train models
tg_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

egc_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

tg_model.fit(X_tg, y_tg)
egc_model.fit(X_egc, y_egc)

print("Models trained successfully")

Models trained successfully


In [8]:

tg_test = test[test["target_type"] == "tg"]
egc_test = test[test["target_type"] == "egc"]

# Convert to TF-IDF features
X_tg_test = vectorizer.transform(tg_test["smiles"])
X_egc_test = vectorizer.transform(egc_test["smiles"])

tg_pred = tg_model.predict(X_tg_test)
egc_pred = egc_model.predict(X_egc_test)

print("TG Predictions:", len(tg_pred))
print("EGC Predictions:", len(egc_pred))

TG Predictions: 2763
EGC Predictions: 1352


In [9]:
import pandas as pd

submission = test[["id"]].copy()

submission["target"] = 0.0

submission.loc[
    submission["id"].isin(tg_test["id"]),
    "target"
] = tg_pred

submission.loc[
    submission["id"].isin(egc_test["id"]),
    "target"
] = egc_pred

submission.to_csv("submission.csv", index=False)

print(submission.shape)
submission.head()

(4115, 2)


,id,target
0,1,278.320600
1,2,4.779613
2,3,73.782300
3,4,46.525000
4,5,109.062150


In [10]:
try:
    import catboost
    print("CatBoost:", catboost.__version__)
except Exception:
    print("CatBoost not available")

try:
    import xgboost
    print("XGBoost:", xgboost.__version__)
except Exception:
    print("XGBoost not available")

try:
    import lightgbm
    print("LightGBM:", lightgbm.__version__)
except Exception:
    print("LightGBM not available")

CatBoost: 1.2.10
XGBoost: 3.2.0
LightGBM: 4.6.0


In [11]:
try:
    import catboost
    print("CatBoost:", catboost.__version__)
except Exception:
    print("CatBoost not available")

try:
    import xgboost
    print("XGBoost:", xgboost.__version__)
except Exception:
    print("XGBoost not available")

try:
    import lightgbm
    print("LightGBM:", lightgbm.__version__)
except Exception:
    print("LightGBM not available")

CatBoost: 1.2.10
XGBoost: 3.2.0
LightGBM: 4.6.0
